# Recover CPGs for rows with missing graphs

This notebook retries only rows whose CPG has no nodes. It adds an explicit, versioned prelude for common **attribute-only** macros during Joern parsing, while retaining the original function text in every result. It writes per-split recovery journals under `output/local_full/joern_macro_recovery_20261031/`; it does not overwrite source CPG JSONL files. Review the recovery journal before merging.

Run this notebook in the existing WSL `.venv-wsl` kernel. Joern retries can take several hours.

In [1]:
from pathlib import Path
import sys, os, json, hashlib, re
from collections import Counter

ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified')
if not ROOT.exists():
    raise FileNotFoundError(f'Project not found: {ROOT}. Run this notebook in WSL.')
sys.path.insert(0, str(ROOT))
from prepare_joern_cpg import extract_batch

WORK = ROOT / 'output/local_full'
REPORT = WORK / 'joern_macro_recovery_20261031'
REPORT.mkdir(parents=True, exist_ok=True)
JOERN_HOME = Path('/home/yungn/tools/joern/y/joern-cli')
JAVA_OPTS = os.environ.get('JOERN_JAVA_OPTS', '-Xms128m -Xmx1536m')
DATASETS = ('devign', 'reveal')
SPLITS = ('train', 'validation', 'test')
print('Project:', ROOT, '| Joern:', JOERN_HOME, '| Report:', REPORT)
assert (JOERN_HOME / 'joern-parse').is_file()

Project: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified | Joern: /home/yungn/tools/joern/y/joern-cli | Report: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/joern_macro_recovery_20261031


## Parse prelude

The defaults below cover tokens used as compiler attributes or parameter annotations. They expand to nothing **only in the temporary Joern input**. We deliberately do not erase `OPPROTO`, `STATUS_PARAM`, `G_GNUC_PRINTF`, `GCC_FMT_ATTR`, or function-like macros: those can change declarations, parameters, or function bodies. Add a macro only after checking its definition in the original project/version. A graph produced after an attribute stub still needs method/body review.

In [2]:
ATTRIBUTE_MACROS = (
    'av_cold', 'coroutine_fn', 'always_inline', 'av_always_inline',
    'av_noinline', 'QEMU_NORETURN', 'attribute_align_arg', 'av_unused',
    'CUDAAPI', 'WINAPI', 'G_GNUC_UNUSED', 'av_flatten',
)
PRELUDE = ''.join(f'#ifndef {name}\n#define {name}\n#endif\n' for name in ATTRIBUTE_MACROS)
BATCH_SIZE = 10  # Joern input files per batch; lower this if memory is constrained.

def sha(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def read_missing(path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if not row.get('nodes'):
                rows.append(row)
    return rows

plan = {}
for dataset in DATASETS:
    for split in SPLITS:
        source = WORK / dataset / 'cpg' / f'{split}.jsonl'
        for row in read_missing(source):
            plan[row['id']] = {'dataset': dataset, 'split': split, 'func_sha256': sha(row['func'])}
print('Missing rows:', len(plan))
print(Counter((v['dataset'], v['split']) for v in plan.values()))

Missing rows: 3637
Counter({('reveal', 'train'): 2020, ('devign', 'train'): 901, ('reveal', 'validation'): 263, ('reveal', 'test'): 233, ('devign', 'test'): 120, ('devign', 'validation'): 100})


## Retry missing rows

This cell is resumable: it skips IDs already present in a split's `results.jsonl` journal. Each new result is flushed to disk immediately. `recovered` means Joern returned a source-function AST under the current filter; it does not certify semantic completeness. Failed rows retain the parse error for later classification.

In [3]:
def completed_ids(journal):
    done = set()
    if journal.exists():
        with journal.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    done.add(json.loads(line)['id'])
    return done

def retry_split(dataset, split):
    source = WORK / dataset / 'cpg' / f'{split}.jsonl'
    journal = REPORT / f'{dataset}_{split}.results.jsonl'
    missing = read_missing(source)
    done = completed_ids(journal)
    pending = [r for r in missing if r['id'] not in done]
    print(f'{dataset}/{split}: missing={len(missing)} done={len(done)} pending={len(pending)}', flush=True)
    with journal.open('a', encoding='utf-8', newline='\n') as out:
        for start in range(0, len(pending), BATCH_SIZE):
            batch_rows = pending[start:start+BATCH_SIZE]
            batch = [(start+i, {**r, 'func': PRELUDE + r['func']}) for i, r in enumerate(batch_rows)]
            try:
                results = extract_batch(batch, JOERN_HOME, JAVA_OPTS)
            except Exception as exc:
                results = [(i, row, {'nodes': [], 'edges': []}, f'{type(exc).__name__}: {str(exc)[:500]}')
                           for i, row in batch]
            for ordinal, transformed, graph, error in results:
                original = batch_rows[ordinal-start]
                record = {
                    'id': original['id'], 'dataset': dataset, 'split': split,
                    'func_sha256': sha(original['func']), 'prelude_sha256': sha(PRELUDE),
                    'macro_prelude': list(ATTRIBUTE_MACROS),
                    'status': 'recovered' if graph.get('nodes') else 'still_missing',
                    'error': error, 'original_func': original['func'],
                    'nodes': graph.get('nodes', []), 'edges': graph.get('edges', []),
                }
                out.write(json.dumps(record, ensure_ascii=False) + '\n')
                out.flush(); os.fsync(out.fileno())
            print(f'  processed {min(start+BATCH_SIZE,len(pending))}/{len(pending)}', flush=True)
    return journal

journals = []
for dataset in DATASETS:
    for split in SPLITS:
        journals.append(retry_split(dataset, split))
print('Journals:', *journals, sep='\n')

devign/train: missing=901 done=0 pending=901
  processed 10/901
  processed 20/901
  processed 30/901
  processed 40/901
  processed 50/901
  processed 60/901
  processed 70/901
  processed 80/901
  processed 90/901
  processed 100/901
  processed 110/901
  processed 120/901
  processed 130/901
  processed 140/901
  processed 150/901
  processed 160/901
  processed 170/901
  processed 180/901
  processed 190/901
  processed 200/901
  processed 210/901
  processed 220/901
  processed 230/901
  processed 240/901
  processed 250/901
  processed 260/901
  processed 270/901
  processed 280/901
  processed 290/901
  processed 300/901
  processed 310/901
  processed 320/901
  processed 330/901
  processed 340/901
  processed 350/901
  processed 360/901
  processed 370/901
  processed 380/901
  processed 390/901
  processed 400/901
  processed 410/901
  processed 420/901
  processed 430/901
  processed 440/901
  processed 450/901
  processed 460/901
  processed 470/901
  processed 480/901
  pr

## Summarize and review

Recovered graphs remain separate from the frozen inputs. Check method names against each original function and inspect rows with parser errors or suspiciously small graphs before any merge. The retry count is not a completeness score.

In [4]:
summary = Counter()
for journal in journals:
    counts = Counter()
    with journal.open(encoding='utf-8') as f:
        for line in f:
            row = json.loads(line); counts[row['status']] += 1
            summary[(row['dataset'], row['split'], row['status'])] += 1
    print(journal.name, dict(counts))
print('Overall:', dict(summary))

devign_train.results.jsonl {'recovered': 790, 'still_missing': 111}
devign_validation.results.jsonl {'recovered': 90, 'still_missing': 10}
devign_test.results.jsonl {'still_missing': 12, 'recovered': 108}
reveal_train.results.jsonl {'still_missing': 2018, 'recovered': 2}
reveal_validation.results.jsonl {'still_missing': 262, 'recovered': 1}
reveal_test.results.jsonl {'still_missing': 232, 'recovered': 1}
Overall: {('devign', 'train', 'recovered'): 790, ('devign', 'train', 'still_missing'): 111, ('devign', 'validation', 'recovered'): 90, ('devign', 'validation', 'still_missing'): 10, ('devign', 'test', 'still_missing'): 12, ('devign', 'test', 'recovered'): 108, ('reveal', 'train', 'still_missing'): 2018, ('reveal', 'train', 'recovered'): 2, ('reveal', 'validation', 'still_missing'): 262, ('reveal', 'validation', 'recovered'): 1, ('reveal', 'test', 'still_missing'): 232, ('reveal', 'test', 'recovered'): 1}
